# Prepare derived synthetic patient records for *Clinical Synopsis: Source-Grounded Patient Summaries*

This notebook prepares the derived patient records used as input for the Clinical Synopsis knowledge base:

```text
        Raw synthetic mCODE/Synthea FHIR records
                        ↓
            Select a 50-patient cohort
                        ↓
Transform records into patient-level Markdown and CSV files
                        ↓
   Write derived records to data/derived/sample50/
```

## Contents

1. [Source dataset](#1-source-dataset)
2. [Select the 50-patient cohort](#2-select-the-50-patient-cohort)
3. [Extract patient-level tabular data](#3-extract-patient-level-tabular-data)
4. [Create derived patient records](#4-create-derived-patient-records)

## 1. Source dataset

We are interested in the **oncology** setting. We're using **synthetic patient data** from generated by Synthea, specifically, the **mCODE test data** built from Synthea because it includes both cancer-related and cancer-unrelated **broader record of a patient's history**, including non-cancer **encounters** (i.e., any professional interactions between a patient and a healthcare provider), **conditions**, and **medications**.

The FHIR format for **Electronic Health Records (EHRs)** is the standard way to represent structured clinical data, which makes the synthetic oncology records in this project realistic and easier to process and reuse across systems. 

**mCODE** (Minimal Common Oncology Data Elements) is an open-source, HL7 FHIR-based data standard for cancer patient data, developed by the American Society of Clinical Oncology (ASCO) and the MITRE Corporation.

We can obtain standardized, synthetic patient data from the HL7 Confluence repository for mCODE Test Data. "Because of the way that Synthea outputs FHIR records, it is not possible at this time to output mCODE patients directly out of Synthea. So these patients have been post-processed using the fhir-mapper."
https://confluence.hl7.org/spaces/COD/pages/80119851/mCODE+Test+Data
(Not yet available are pathologic staging, genetics/genomics, metastasis records.)


For this project we selected 50 patients from the **STU2 breast cancer, lifetime/longitudinal EERs dataset** using a fixed, step-by-step rule based on medical complexity (how often a patient was seen, i.e., "encounter count", and how long they were followed, i.e., "follow-up time"). See `data/processed/mcode_breast_sample_50_manifest.csv` for details. (The breast cancer dataset from mCode is more densely populated than the mixed-cancer set, because it provides broad patient history.)

Our challenge is not only retrieval, but a source-grounded synthesis across mixed document types. (Which makes our evalution focus on patient-summary quality and source-grounding.) Therefore, for each patient we want mixed documents such as encounter summaries, lab results, medication lists, and oncology-related reports, so that the RAG application retrieves from both text and structured metadata (i.e., Markdown + CSV).

We turn those records into turn into a 50 patient corpus with a small metadata database. We do not want the the LLM synthesize everything from raw tables for every queary, so in the final RAG pipeline, we treat `patient_overview.md` generated for each patient as the trusted summary source, while the CSVs are the sources of additional details.


## 2. Select the 50-patient cohort

The script `clinical_synopsis/scripts/sample_mcode_patients.py` uses the raw mCODE breast cancer patient json bundles in `data/raw/longitudinalMCODEBreast` to compute per‑patient statistics (counts of encounters, observations, conditions, follow‑up duration, a “complexity” score), assigns each patient to a **low/medium/high complexity** bucket, and then draws a stratified random sample of 50 patients. The full stats table and the 50‑patient manifest are saved as CSV files in `data/processed`.

```bash
uv run clinical_synopsis/scripts/sample_mcode_patients.py
```
Output:
```
Found 258 JSON files total
Kept 256 patient bundles
Skipped 2 non-patient bundles

Wrote patient stats to: data/processed/mcode_breast_patient_stats.csv
Wrote 50-patient manifest to: data/processed/mcode_breast_sample_50_manifest.csv

Sample bucket counts:
complexity_bucket
low       17
high      17
medium    16
```

`create_prototype_buckets.py` reads the manifest `data/processed/mcode_breast_sample_50_manifest.csv`, copies the selected 50 JSON files into `data/prototype/sample50`, and also creates low, medium, and high folders.

```bash
uv run clinical_synopsis/scripts/create_prototype_buckets.py
```
Output:
```
Copied 50 files.

Bucket counts in manifest:
complexity_bucket
low       17
high      17
medium    16

Output folders:
- data/prototype/sample50
- data/prototype/low
- data/prototype/medium
- data/prototype/high
```

In [2]:
!find ../data/prototype/high -type f | wc -l

17


### Note on COMPLEXITY SCORES for each patient

A higher complexity score should reflect more encounters, conditions, procedures, meds, reports for a given patient, as well as a longer follow-up period (e.g., Febrile neutropenia condition gives a clear date (onsetDateTime, recordedDate) and is tied to an encounter, which contributes to follow-up and complexity).

In `sample_mcode_patients.py` a complexity score is computed as:
```python
complexity_score = (
    counts["Encounter"] * 3
    + counts["Observation"] * 1
    + counts["Condition"] * 2
    + counts["Procedure"] * 2
    + counts["MedicationRequest"] * 2
    + counts["MedicationAdministration"] * 2
    + counts["DiagnosticReport"] * 2
    + min(followup_days // 180, 20)
)
```
Which means that:
- Each resource type contributes with a **weight**:
  - Encounters: \(3 \times\) number of encounters (heavier weight).
  - Observations: \(1 \times\) number of observations.
  - Conditions, Procedures, MedicationRequest, MedicationAdministration, DiagnosticReport: each \(2 \times\) their counts.
- Plus a **time component**:
  - `followup_days` is the difference between the first and last clinical dates found in the bundle.
  - `followup_days // 180` converts follow-up into “half-year blocks”.
  - This term is capped at 20, so very long records don’t dominate.



## 3. Extract patient-level tabular data

`extract_mcode_patient_tables.py` extracts each sampled FHIR bundle into a per-patient folder of CSV and JSON files. For each patient in `data/prototype/sample50` it creates:
```
data/interim/sample50/<patient_id>/patient.csv
.../encounters.csv
.../conditions.csv
.../observations.csv
.../medication_requests.csv
.../medication_administrations.csv
.../procedures.csv
.../diagnostic_reports.csv
.../bundle_metadata.json
```



```bash
uv run clinical_synopsis/scripts/extract_mcode_patient_tables.py
```
Output:
```
Processed 50 patient files into data/interim/sample50
```

## 4. Create derived patient records

`generate_derived_documents.py` creates the documents that the RAG searches over, i.e., the mock health records imitating the structure and content of real clinical records.

It uses the following logic for each patient:
- Always generate:
    - patient_overview.md
    - encounters.csv
    - conditions.csv
    - observations.csv
- Generate conditionally:
    - medications.csv if medication rows exist.
    - procedures.csv if procedure rows exist.
    - diagnostic_reports.csv if report rows exist.
    - oncology_timeline.md if there are at least 3 dated oncology-related events (very simple key word matching, so fine just for protoyping. Why 3? With only 1 event, there is no “timeline” — just a single date; with 2, you only have a start and one follow‑up, which is not longitudinal.)


Note that in patient_overview.md “recent” means top=N entries based on the available dates:
  - N = 10 for conditions, medication, procedures, reports
  - N = 12 for observations
  - N = 8 for encounters
For a real-life application there should be instead an explicit time window** (like “last year”).


```bash
uv run clinical_synopsis/scripts/generate_derived_documents.py 
```
Output:
```
Processed 50 patients into data/derived/sample50
```
